# Scenario 1 analysis — local JGEA output

Notebook này chạy local, đọc file `out/Scenario_1.csv` sinh từ JGEA. Nó chuẩn hóa cột từ schema JGEA hiện tại (`n.evals`, `best→quality`, ...) sang schema phân tích (`evals`, `best_fitness`, ...), rồi xuất các hình và bảng tương tự notebook gốc của tác giả.


In [ ]:
import os, glob, math, statistics
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon

try:
    import matplotlib_inline.backend_inline
    matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
except Exception:
    pass


In [ ]:
# =========================
# Config local
# =========================
experiment_name = 'Scenario_1'
separator = ';'
delta_quantization = 100
level_significance = 0.01
efficiency_percentile_reference = 0.75

# File CSV của bạn. Ưu tiên out/Scenario_1.csv.
candidates = [
    'out/Scenario_1.csv',
    './out/Scenario_1.csv',
    'Scenario_1.csv',
    'out/scenario1-progress.csv',
    'scenario1-progress.csv',
]
file_name = next((f for f in candidates if os.path.exists(f)), None)
if file_name is None:
    found = glob.glob('*.csv') + glob.glob('out/*.csv')
    raise FileNotFoundError(f'Không thấy CSV Scenario 1. CSV tìm được: {found}')

# Output folders
os.makedirs(f'Figures/{experiment_name}', exist_ok=True)
os.makedirs(f'LaTeX/{experiment_name}', exist_ok=True)
os.makedirs(f'XLSX Files/{experiment_name}', exist_ok=True)

print('Reading:', file_name)
fileCSV = pd.read_csv(file_name, sep=separator)
print('Shape:', fileCSV.shape)
fileCSV.head()


In [ ]:
# =========================
# Chuẩn hóa cột cho file JGEA local
# =========================
fileCSV = fileCSV.copy()

# Rename các cột JGEA local sang tên notebook gốc kỳ vọng
rename_map = {
    'n.iterations': 'iterations',
    'n.evals': 'evals',
    'n.births': 'births',
    'elapsed.secs': 'elapsed.seconds',
    'all→each[quality]→uniqueness': 'all→each[fitness]→uniqueness',
    'best→quality': 'best_fitness',
    'best→fitness': 'best_fitness',
}
fileCSV = fileCSV.rename(columns={k: v for k, v in rename_map.items() if k in fileCSV.columns})

# Nếu best_fitness chưa có, tự tìm cột quality/fitness phù hợp
if 'best_fitness' not in fileCSV.columns:
    candidates_best = [c for c in fileCSV.columns if ('best' in c and ('quality' in c or 'fitness' in c))]
    if not candidates_best:
        raise ValueError(f'Không tìm được cột best fitness/quality. Columns: {list(fileCSV.columns)}')
    fileCSV = fileCSV.rename(columns={candidates_best[0]: 'best_fitness'})
    print('Using best fitness column:', candidates_best[0])

required = ['seed', 'problem', 'solver_sigma', 'iterations', 'evals', 'births', 'elapsed.seconds', 'best_fitness']
missing = [c for c in required if c not in fileCSV.columns]
if missing:
    raise ValueError(f'Thiếu cột bắt buộc: {missing}\nColumns hiện có: {list(fileCSV.columns)}')

# Kiểu dữ liệu
fileCSV['seed'] = pd.to_numeric(fileCSV['seed'], errors='coerce').astype('Int64')
for c in ['iterations', 'evals', 'births', 'elapsed.seconds', 'best_fitness']:
    fileCSV[c] = pd.to_numeric(fileCSV[c], errors='coerce')

# Tạo solver và sigma robust: cmaEs/de/pso không có sigma; simpleEs-0.02, ga-0.25 có sigma
base_solvers = {'cmaEs', 'de', 'pso'}
def split_solver_sigma(x):
    x = str(x)
    if x in base_solvers:
        return x, x
    if '-' in x:
        solver, sigma = x.split('-', 1)
        return solver, sigma
    return x, x

split = fileCSV['solver_sigma'].apply(split_solver_sigma)
fileCSV['solver'] = split.apply(lambda t: t[0])
fileCSV['sigma'] = split.apply(lambda t: t[1])
fileCSV['solver_sigma_seed'] = fileCSV['solver_sigma'].astype(str) + '_seed:' + fileCSV['seed'].astype(str)
fileCSV['objective'] = 'minimize'

# eval quantization cho convergence plot
fileCSV['evals_quantized'] = (fileCSV['evals'].astype(int) // delta_quantization)

# Lấy genotype size: ưu tiên cột best→genotype→size, nếu không có thì parse từ tên problem
if 'best→genotype→size' in fileCSV.columns:
    fileCSV['genotype_size'] = pd.to_numeric(fileCSV['best→genotype→size'], errors='coerce').astype('Int64')
elif 'best_genotype_size' in fileCSV.columns:
    fileCSV['genotype_size'] = pd.to_numeric(fileCSV['best_genotype_size'], errors='coerce').astype('Int64')
else:
    fileCSV['genotype_size'] = fileCSV['problem'].astype(str).str.extract(r'-(\d+)$')[0].astype('Int64')

united_CSV = fileCSV.copy()
print('Columns after normalization:')
print(list(united_CSV.columns))
united_CSV.head()


In [ ]:
# =========================
# Order giống paper
# =========================
problem_order = []
for prefix in ['Sphere', 'PA-1', 'PA-10', 'CPA', 'Ackley', 'Rastrigin', 'Griewank', 'Rosenbrock']:
    for p in [20, 100, 200, 500]:
        problem_order.append(f'{prefix}-{p}')
problems = [p for p in problem_order if p in set(united_CSV['problem'])]
# fallback nếu tên khác
if not problems:
    problems = sorted(united_CSV['problem'].drop_duplicates().tolist())

solver_order = ['cmaEs', 'de', 'pso', 'simpleEs-0.02', 'simpleEs-0.25', 'simpleEs-0.5', 'ga-0.02', 'ga-0.25', 'ga-0.5']
solvers_sigmas = [s for s in solver_order if s in set(united_CSV['solver_sigma'])]
if not solvers_sigmas:
    solvers_sigmas = sorted(united_CSV['solver_sigma'].drop_duplicates().tolist())

solvers = [s for s in ['cmaEs','de','pso','simpleEs','ga'] if s in set(united_CSV['solver'])]
if not solvers:
    solvers = sorted(united_CSV['solver'].drop_duplicates().tolist())

sigmas = sorted(united_CSV['sigma'].drop_duplicates().tolist(), key=lambda x: str(x))
seeds = sorted([int(s) for s in united_CSV['seed'].dropna().drop_duplicates().tolist()])
solvers_sigmas_seeds = sorted(united_CSV['solver_sigma_seed'].drop_duplicates().tolist())

print(f'Problems ({len(problems)}):', problems)
print(f'Solver sigmas ({len(solvers_sigmas)}):', solvers_sigmas)
print(f'Seeds ({len(seeds)}):', seeds[:10], '...' if len(seeds) > 10 else '')

# Kiểm tra số run cuối
last_evals_df = united_CSV[united_CSV.groupby(['problem', 'solver_sigma_seed'])['evals'].transform('max') == united_CSV['evals']].copy()
print('Final rows:', len(last_evals_df))
print('Unique final runs:', last_evals_df[['problem','solver_sigma','seed']].drop_duplicates().shape[0])
print('Expected final runs:', len(problems) * len(solvers_sigmas) * len(seeds))
last_evals_df.head()


In [ ]:
# Style
plt.rcParams.update({'font.size': 6, 'lines.linewidth': 0.8})
color_map = mpl.colormaps['tab10']


In [ ]:
################################################################################
# Convergence plots
################################################################################
fig, axs = plt.subplots(len(problems), len(solvers), sharex=True, sharey='row', layout='constrained', figsize=(2.2*len(solvers), max(4, 0.65*len(problems))))
if len(problems) == 1 and len(solvers) == 1:
    axs = np.array([[axs]])
elif len(problems) == 1:
    axs = np.array([axs])
elif len(solvers) == 1:
    axs = np.array([[ax] for ax in axs])

fig.suptitle('Scenario 1 — convergence plots', fontsize=12, fontweight='bold')
fig.supxlabel('x = Evals', fontsize=8, fontweight='bold')
fig.supylabel('y = Best Fitness', fontsize=8, fontweight='bold')

for i, problem in enumerate(problems):
    axs[i, len(solvers)-1].yaxis.set_label_position('right')
    axs[i, len(solvers)-1].set_ylabel(problem, rotation='horizontal', labelpad=10, horizontalalignment='left')
    pre_problem = united_CSV[united_CSV['problem'] == problem]

    for j, solver in enumerate(solvers):
        axs[0, j].set_title(solver, pad=10, rotation=10, weight='bold')
        pre_solver = pre_problem[pre_problem['solver'] == solver]

        # với cmaEs/de/pso, sigma = tên solver; với simpleEs/ga, sigma = số
        solver_sigmas_for_solver = [ss for ss in solvers_sigmas if ss in set(pre_solver['solver_sigma'])]
        for k, solver_sigma in enumerate(solver_sigmas_for_solver):
            color = color_map.colors[k % len(color_map.colors)]
            filtered = pre_solver[pre_solver['solver_sigma'] == solver_sigma]
            if filtered.empty:
                continue
            grouped = filtered.groupby('evals_quantized')['best_fitness'].describe()[['25%', '50%', '75%']]
            evals = grouped.index * delta_quantization
            median = grouped['50%']
            q1 = grouped['25%']
            q3 = grouped['75%']
            axs[i, j].plot(evals, median, color=color, label=solver_sigma)
            axs[i, j].fill_between(evals, q1, q3, color=color, alpha=0.12)
            axs[i, j].grid(True, alpha=0.2)

# legend gọn hơn
legend_elements = [Line2D([0], [0], color=color_map.colors[i % len(color_map.colors)], lw=2, label=s) for i, s in enumerate(solvers_sigmas)]
fig.legend(handles=legend_elements, loc='center right', title='Solver')
plt.savefig(f'Figures/{experiment_name}/{experiment_name}_Convergence.png', bbox_inches='tight', transparent=False, dpi=300)
plt.show()


In [ ]:
################################################################################
# Effectiveness boxplots — final best fitness, lower is better
################################################################################
fig, axs = plt.subplots(1, len(problems), sharex=False, sharey=True, layout='constrained', figsize=(2.0*len(problems), 0.55*len(solvers_sigmas)))
if len(problems) == 1:
    axs = [axs]
fig.suptitle('Scenario 1 — final best fitness (lower is better)', fontsize=12, fontweight='bold')
fig.supxlabel('x = Best Fitness', fontsize=8, fontweight='bold')

big_title = experiment_name + '_EffectiveMatrix_'

for i, problem in enumerate(problems):
    axs[i].set_title(problem, pad=10, weight='bold', fontsize=7)
    data_to_plot, labels = [], []
    for solver_sigma in solvers_sigmas:
        vals = last_evals_df[(last_evals_df['problem'] == problem) & (last_evals_df['solver_sigma'] == solver_sigma)]['best_fitness'].dropna().tolist()
        if vals:
            data_to_plot.append(vals)
            labels.append(solver_sigma)
    if not data_to_plot:
        axs[i].axis('off')
        continue
    bp = axs[i].boxplot(data_to_plot, patch_artist=True, vert=False, labels=labels)

    # Save data for LaTeX plots; pad uneven arrays just in case
    max_len = max(len(x) for x in data_to_plot)
    padded = [x + [np.nan]*(max_len-len(x)) for x in data_to_plot]
    df_latex = pd.DataFrame(dict(zip(labels, padded)))
    df_latex.to_csv(f'LaTeX/{experiment_name}/{big_title}{problem}.txt', sep='\t', index=False)

    for patch, color in zip(bp['boxes'], color_map.colors[:len(labels)]):
        patch.set_facecolor(color)
    axs[i].grid(True, axis='x', alpha=0.25)

plt.savefig(f'Figures/{experiment_name}/{experiment_name}_Effectiveness.png', bbox_inches='tight', transparent=False, dpi=300)
plt.show()


In [ ]:
################################################################################
# Number of Victories Score (NoVS)
################################################################################
binary_ranking_effectiveness_df = pd.DataFrame(index=problems, columns=solvers_sigmas, dtype=object)

for problem in problems:
    filtered = last_evals_df[last_evals_df['problem'] == problem]
    if filtered.empty:
        continue

    # lower is better
    median_data = filtered.groupby('solver_sigma')['best_fitness'].median().sort_values(ascending=True)
    best_solver_sigma = median_data.index[0]
    best_df = filtered[filtered['solver_sigma'] == best_solver_sigma].sort_values('seed')['best_fitness'].reset_index(drop=True)

    for solver_sigma in solvers_sigmas:
        if best_solver_sigma == solver_sigma:
            binary_ranking_effectiveness_df.at[problem, solver_sigma] = 1
            continue
        other_df = filtered[filtered['solver_sigma'] == solver_sigma].sort_values('seed')['best_fitness'].reset_index(drop=True)
        if len(other_df) == 0:
            binary_ranking_effectiveness_df.at[problem, solver_sigma] = np.nan
            continue
        # Wilcoxon cần cùng độ dài; nếu thiếu seed thì inner length
        m = min(len(best_df), len(other_df))
        try:
            _, p_value = wilcoxon(best_df.iloc[:m], other_df.iloc[:m])
        except ValueError:
            p_value = 0.0
        binary_ranking_effectiveness_df.at[problem, solver_sigma] = 1 if p_value > level_significance else 0

binary_score_effectiveness = binary_ranking_effectiveness_df.astype(float).sum()
binary_score_effectiveness.name = 'NOVS'
binary_ranking_df = pd.concat([binary_ranking_effectiveness_df, binary_score_effectiveness.to_frame().T])

xlsx_path = f'XLSX Files/{experiment_name}/{experiment_name}_NOVS.xlsx'
tex_path = f'LaTeX/{experiment_name}/{experiment_name}_NOVS.tex'
binary_ranking_df.to_excel(xlsx_path)
with open(tex_path, 'w', encoding='utf-8') as f:
    f.write(binary_ranking_df.to_latex(index=True, escape=False))

print('Saved:', xlsx_path)
print('Saved:', tex_path)
print('NOVS:')
print(binary_ranking_df.loc['NOVS'].sort_values(ascending=False))
binary_ranking_df


In [ ]:
################################################################################
# Normalized Effectiveness Rank (NER)
################################################################################
normalization_factor = len(solvers_sigmas) * len(seeds)
solvers_sigmas__rankings = [[] for _ in range(len(solvers_sigmas))]
ner_rows = []

for problem in problems:
    filtered = last_evals_df[last_evals_df['problem'] == problem].sort_values('best_fitness', ascending=True).reset_index(drop=True)
    for rank_idx, row in filtered.iterrows():
        solver_sigma = row['solver_sigma']
        rank_norm = (rank_idx + 1) / normalization_factor
        if solver_sigma in solvers_sigmas:
            solvers_sigmas__rankings[solvers_sigmas.index(solver_sigma)].append(rank_norm)
        ner_rows.append({
            'problem': problem,
            'solver_sigma': solver_sigma,
            'seed': row['seed'],
            'NER': rank_norm,
        })

NER = pd.DataFrame(ner_rows)

plt.figure(figsize=(6, 3.8))
bp = plt.boxplot(solvers_sigmas__rankings, patch_artist=True, vert=False, labels=solvers_sigmas)
for patch, color in zip(bp['boxes'], color_map.colors[:len(solvers_sigmas)]):
    patch.set_facecolor(color)
plt.title('Scenario 1 — NER (lower is better)', fontsize=12, fontweight='bold')
plt.xlabel('NER')
plt.grid(True, axis='x', alpha=0.25)

# Save data for latex
max_len = max(len(x) for x in solvers_sigmas__rankings)
padded = [x + [np.nan]*(max_len-len(x)) for x in solvers_sigmas__rankings]
df_ner_latex = pd.DataFrame(dict(zip(solvers_sigmas, padded)))
df_ner_latex.to_csv(f'LaTeX/{experiment_name}/AAA_{experiment_name}_NER.txt', sep='\t', index=False)
NER.to_excel(f'XLSX Files/{experiment_name}/{experiment_name}_NER_raw.xlsx', index=False)

plt.savefig(f'Figures/{experiment_name}/{experiment_name}_NER.png', bbox_inches='tight', transparent=False, dpi=300)
plt.show()
NER.head()


In [ ]:
################################################################################
# Efficiency: evals to 3rd quartile (EtTQ)
################################################################################
solvers_sigmas__first_eval_to_hit_threshold = [[] for _ in range(len(solvers_sigmas))]
ettq_rows = []

for problem in problems:
    final_problem = last_evals_df[last_evals_df['problem'] == problem].sort_values('best_fitness', ascending=True).reset_index(drop=True)
    if final_problem.empty:
        continue
    idx = int(efficiency_percentile_reference * len(final_problem))
    idx = min(idx, len(final_problem)-1)
    threshold = final_problem.iloc[idx]['best_fitness']
    problem_data = united_CSV[united_CSV['problem'] == problem]
    max_budget = int(problem_data['evals'].max())

    for solver_sigma in solvers_sigmas:
        solver_data = problem_data[problem_data['solver_sigma'] == solver_sigma]
        for seed in seeds:
            seed_data = solver_data[solver_data['seed'].astype(int) == int(seed)].sort_values('evals')
            if seed_data.empty:
                continue
            hit = seed_data[seed_data['best_fitness'] <= threshold]
            ettq = int(hit.iloc[0]['evals']) if not hit.empty else max_budget
            solvers_sigmas__first_eval_to_hit_threshold[solvers_sigmas.index(solver_sigma)].append(ettq)
            ettq_rows.append({
                'problem': problem,
                'solver_sigma': solver_sigma,
                'seed': seed,
                'EtTQ': ettq,
                'threshold': threshold,
            })

EtTQ = pd.DataFrame(ettq_rows)

plt.figure(figsize=(6, 3.8))
bp = plt.boxplot(solvers_sigmas__first_eval_to_hit_threshold, patch_artist=True, vert=False, labels=solvers_sigmas)
for patch, color in zip(bp['boxes'], color_map.colors[:len(solvers_sigmas)]):
    patch.set_facecolor(color)
plt.title('Scenario 1 — EtTQ (lower is better)', fontsize=12, fontweight='bold')
plt.xlabel('Evaluations to third quartile')
plt.grid(True, axis='x', alpha=0.25)

# Save data for LaTeX/XLSX
max_len = max(len(x) for x in solvers_sigmas__first_eval_to_hit_threshold)
padded = [x + [np.nan]*(max_len-len(x)) for x in solvers_sigmas__first_eval_to_hit_threshold]
df_ettq_latex = pd.DataFrame(dict(zip(solvers_sigmas, padded)))
df_ettq_latex.to_csv(f'LaTeX/{experiment_name}/AAA_{experiment_name}_EtTQ.txt', sep='\t', index=False)
EtTQ.to_excel(f'XLSX Files/{experiment_name}/{experiment_name}_EtTQ_raw.xlsx', index=False)

plt.savefig(f'Figures/{experiment_name}/{experiment_name}_EtTQ.png', bbox_inches='tight', transparent=False, dpi=300)
plt.show()
EtTQ.head()


In [ ]:
################################################################################
# Mean NER vs genotype size p
################################################################################
mean_NER = pd.DataFrame(index=problems, columns=solvers_sigmas, dtype=float)

for problem in problems:
    filtered = last_evals_df[last_evals_df['problem'] == problem].sort_values('best_fitness', ascending=True).reset_index(drop=True)
    rankings_by_solver = {s: [] for s in solvers_sigmas}
    for rank_idx, row in filtered.iterrows():
        rankings_by_solver[row['solver_sigma']].append((rank_idx + 1) / (len(solvers_sigmas) * len(seeds)))
    for solver_sigma in solvers_sigmas:
        vals = rankings_by_solver.get(solver_sigma, [])
        mean_NER.loc[problem, solver_sigma] = statistics.mean(vals) if vals else np.nan

genotype_size_series = last_evals_df.groupby('problem')['genotype_size'].first()
mean_NERs = pd.concat([genotype_size_series, mean_NER], axis=1)
mean_NERs.to_excel(f'XLSX Files/{experiment_name}/{experiment_name}_meanNER-vs-p.xlsx')

mean_NERs_sorted = mean_NERs.sort_values(by='genotype_size')
mean_NERs_grouped = mean_NERs_sorted.groupby('genotype_size').mean(numeric_only=True).reset_index()
mean_NERs_grouped.to_csv(f'LaTeX/{experiment_name}/AAA_{experiment_name}_meanNER-vs-p.txt', sep='\t', index=False)

plt.figure(figsize=(5.5, 3.5))
for i, solver_sigma in enumerate(solvers_sigmas):
    plt.plot(mean_NERs_grouped['genotype_size'], mean_NERs_grouped[solver_sigma], marker='o', label=solver_sigma)
plt.title('Scenario 1 — mean NER vs p', fontsize=12, fontweight='bold')
plt.xlabel('p / genotype size')
plt.ylabel('Mean NER')
plt.grid(True, alpha=0.25)
plt.legend(fontsize=6, ncol=2)
plt.savefig(f'Figures/{experiment_name}/{experiment_name}_meanNER-vs-p.png', bbox_inches='tight', transparent=False, dpi=300)
plt.show()
mean_NERs_grouped


In [ ]:
# Summary of generated files
print('Generated figures:')
for f in sorted(glob.glob(f'Figures/{experiment_name}/*')):
    print(' ', f)
print('\nGenerated XLSX:')
for f in sorted(glob.glob(f'XLSX Files/{experiment_name}/*')):
    print(' ', f)
print('\nGenerated LaTeX/text data:')
for f in sorted(glob.glob(f'LaTeX/{experiment_name}/*'))[:20]:
    print(' ', f)
